In [20]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RepeatedStratifiedKFold, GridSearchCV, KFold
from sksurv.util import Surv
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored, integrated_brier_score
from sksurv.nonparametric import kaplan_meier_estimator
from sklearn.model_selection import StratifiedKFold


In [21]:
seed = 42
time_for_eval = 24
quick = True

R, K = (1, 5) if quick else (5, 5)
n_trees = 200 if quick else 500

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

data = pd.read_csv("data/acrg_clean.csv")
features = ["age", "male", "serosa", "n_level"]
data["male"] = (data["sex"] == "Male").astype(float)
X = data[features].astype(float)
y = Surv.from_arrays(event=data["event"].astype(bool).values, time=data["time_months"].values)
print(X.shape, "events:", int(data["event"].sum()))

(252, 4) events: 103


In [22]:
def c_index_time_for_eval(y_true, risk, tfe = time_for_eval):
    event_tfe = y_true["event"] & (y_true["time"] < tfe)
    return concordance_index_censored(event_tfe, y_true["time"], risk)[0]

def eval_times(y_test, tfe = time_for_eval):
    hi = min(tfe, y_test["time"].max() - 1e-6)
    lo = max(1.0, y_test["time"].min() + 1e-6)
    return np.linspace(lo, hi, 24)

def surv_at(model, X_, times):
    return np.vstack([fn(times) for fn in model.predict_survival_function(X_)])

def km_null_surv(y_train, times, n):
    t, s = kaplan_meier_estimator(y_train["event"], y_train["time"])
    s_at = np.array([s[t <= u][-1] if (t <= u).any() else 1.0 for u in times])
    return np.tile(s_at, (n, 1))

In [23]:
def time_of_eval_scorer(estimator, X_, y_):
    return c_index_time_for_eval(y_, estimator.predict(X_))

def inner_cv(y_tr):
    return list(StratifiedKFold(3, shuffle=True, random_state=seed).split(np.zeros(len(y_tr)), y_tr["event"]))

def make_cox(y_tr):
    pipe = Pipeline([("scale", StandardScaler()),
                     ("cox", CoxnetSurvivalAnalysis(fit_baseline_model=True))])
    grid = {"cox__alphas": [[1e-3], [1e-2], [1e-1], [1.0]], "cox__l1_ratio": [0.1, 0.5, 0.9]}
    return GridSearchCV(pipe, grid, cv=inner_cv(y_tr), scoring=time_of_eval_scorer, error_score=0.5)

def make_rsf(y_tr):
    rsf = RandomSurvivalForest(n_estimators=n_trees, random_state=seed, n_jobs=-1)
    grid = {"min_samples_leaf": [5, 10, 20, 40], "max_features": [1, 2, 4]}
    return GridSearchCV(rsf, grid, cv=inner_cv(y_tr), scoring=time_of_eval_scorer)

models = {"Cox": make_cox, "RSF": make_rsf}

In [24]:
rows, out_of_fold = [], []
cv = RepeatedStratifiedKFold(n_splits=K, n_repeats=R, random_state=seed)
t0 = time.time()

for split, (tr, te) in enumerate (cv.split(X, y["event"])):
    rep, fold = divmod(split, K)
    X_tr, X_te, y_tr, y_te = X.iloc[tr], X.iloc[te], y[tr], y[te]
    times = eval_times(y_te)

    risk0 = (X_te["serosa"] + X_te["n_level"]).values
    rows.append({"rep": rep, "fold": fold, "model": "Stage only", "c_tau": c_index_time_for_eval(y_te, risk0), "ibs": np.nan, "params": ""})
    rows.append({"rep": rep, "fold": fold, "model": "KM (no predictors)", "c_tau": 0.5,
                 "ibs": integrated_brier_score(y_tr, y_te, km_null_surv(y_tr, times, len(te)), times), "params": ""})

    for name, make in models.items():
        search = make(y_tr).fit(X_tr, y_tr)
        best = search.best_estimator_
        risk = best.predict(X_te)
        S = surv_at(best, X_te, times)
        rows.append({"rep": rep, "fold": fold, "model": name, "c_tau": c_index_time_for_eval(y_te, risk),
                     "ibs": integrated_brier_score(y_tr, y_te, S, times), "params": str(search.best_params_)})
        out_of_fold.append(pd.DataFrame({"rep": rep, "model": name, "patient_id": data["patient_id"].iloc[te].values,
                                 "risk": risk, "surv_tau": S[:, -1]}))
    print(f"split {split+1}/{R*K} done ({time.time()-t0:.0f}s)")

res = pd.DataFrame(rows)
out_of_fold = pd.concat(out_of_fold, ignore_index=True)
res.to_csv(results_dir / "e1_acrg_folds.csv", index=False)
out_of_fold.to_csv(results_dir / "e1_acrg_oof.csv", index=False)

C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step

split 1/5 done (8s)


C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step

split 2/5 done (17s)


C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step

split 3/5 done (25s)


C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step

split 4/5 done (33s)


C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step

split 5/5 done (41s)


In [26]:
summary = res.groupby("model")[["c_tau", "ibs"]].agg(["mean", "std"]).round(3)
display(summary)

def corrected_ttest(d, n_train, n_test, alpha=0.05):
    d = np.asarray(d, float); J = len(d)
    se = np.sqrt((1 / J + n_test / n_train) * d.var(ddof=1))
    t = d.mean() / se
    p = 2 * stats.t.sf(abs(t), df=J - 1)
    q = stats.t.ppf(1 - alpha / 2, df=J - 1)
    return d.mean(), d.mean() - q * se, d.mean() + q * se, p

n_test = len(X) // K; n_train = len(X) - n_test
wide = res.pivot_table(index=["rep", "fold"], columns="model", values="c_tau")
EPS = 0.05
for m in ["RSF", "Stage only"]:
    d = wide[m] - wide["Cox"]
    mean, lo95, hi95, p = corrected_ttest(d, n_train, n_test)
    _, lo90, hi90, _ = corrected_ttest(d, n_train, n_test, alpha=0.10)
    equiv = (lo90 > -EPS) and (hi90 < EPS)
    print(f"{m} - Cox: {mean:+.3f}  95% CI [{lo95:+.3f}, {hi95:+.3f}]  p={p:.2f}  equivalent within ±{EPS}: {equiv}")

c_tau           ibs       
                     mean    std   mean    std
model                                         
Cox                 0.680  0.056  0.146  0.018
KM (no predictors)  0.500  0.000  0.163  0.017
RSF                 0.683  0.066  0.147  0.019
Stage only          0.686  0.062    NaN    NaN

RSF - Cox: +0.003  95% CI [-0.033, +0.038]  p=0.84  equivalent within ±0.05: True
Stage only - Cox: +0.005  95% CI [-0.035, +0.046]  p=0.73  equivalent within ±0.05: True
